# Lab 7 — Troubleshoot, monitor, optimize + compute (Exam sections 6 · 10% and 1 · 6%)

**How this lab works**
- Run cells top to bottom (`Shift+Enter`). After a kernel restart, run again from the top.
- Cells marked `# TODO` contain `<FILL-IN>`: replace it with your code, then run the `check(...)` cell.
- Stuck? Ask Claude in the terminal: *"hint for 1c"*. Solutions are in `notebooks/solutions/` (try first!).
- 🖥 **Local** means open-source Spark + Delta on your Mac. ☁ **Databricks** shows the real syntax for the exam.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # lets the notebook import labutils
from labutils import get_spark, path, reset_lab, show, checkpoint, table_path, landing
from labutils.check import check
from pyspark.sql import functions as F
spark = get_spark()
reset_lab("lab7")
from labutils.submit import submit
# 90% of the sales belong to ONE customer -> a "hot key"
sales = (spark.range(200000)
    .withColumn("customer_id", F.when(F.col("id") % 10 < 9, F.lit(1)).otherwise(F.col("id") % 50))
    .withColumn("amount", (F.col("id") % 7 + 1).cast("double")))
sales.write.mode("overwrite").saveAsTable("lab7.sales")

## 7a · Find data skew and fix it with salting

In [ ]:
sales = spark.table("lab7.sales")
by_key = sales.repartition(8, "customer_id").groupBy(F.spark_partition_id().alias("task")).count().orderBy("task")
print("Rows per task when we shuffle by customer_id (like a groupBy):")
show(by_key)

In [ ]:
# TODO 7a
SALTS = 8
salted = sales.withColumn("salt", (F.rand(seed=1) * SALTS).cast("int"))
# step 1: partial sums per (customer_id, salt) -> the hot key is split into 8 pieces
partial = salted.groupBy(<FILL-IN 1>).agg(F.sum("amount").alias("partial_amount"))
# step 2: combine the pieces -> one total per customer
totals = partial.groupBy(<FILL-IN 2>).agg(F.sum(<FILL-IN 3>).alias("total_amount"))
partial.write.mode("overwrite").saveAsTable("lab7.partial_totals")
totals.write.mode("overwrite").saveAsTable("lab7.customer_totals")
print("Rows per task when we shuffle by customer_id + salt:")
show(salted.repartition(8, "customer_id", "salt").groupBy(F.spark_partition_id().alias("task")).count().orderBy("task"))

In [ ]:
check("7a")

## 7b · Liquid clustering

In [ ]:
# TODO 7b
spark.sql("""<FILL-IN 1>""")
spark.sql("INSERT INTO lab7.events SELECT id, date_add(DATE'2024-01-01', CAST(id % 30 AS INT)), id % 50 FROM range(5000)")
spark.sql("""<FILL-IN 2>""")
spark.sql("""<FILL-IN 3>""")
print("clustering columns:", spark.sql("DESCRIBE DETAIL lab7.events").first().clusteringColumns)

In [ ]:
check("7b")

## 7c · Diagnose failing jobs

In [ ]:
ERRORS = {
  "E1": "java.lang.OutOfMemoryError: Java heap space ... at Dataset.collect / toPandas() on the DRIVER",
  "E2": "ModuleNotFoundError / ImportError: numpy.core.multiarray failed to import (after installing a new library version on the cluster)",
  "E3": "Cluster terminated. Reason: Cloud provider launch failure — instance quota exceeded / init script exited with non-zero code",
  "E4": "Task status: Timed Out after 45 minutes (data volume grew; the task now needs ~60 minutes)",
  "E5": "Spark UI: one task 10 min, others 30 s; Max shuffle read 5 GB vs median 400 MB; 'Spill (disk)' is large",
}
FIXES = {
  "A": "Raise the task's timeout threshold above 60 minutes (with headroom)",
  "B": "Don't collect big data to the driver: write to a table / aggregate first, or use a bigger driver",
  "C": "Fix or pin conflicting library versions (cluster/notebook-scoped libraries, requirements), then restart",
  "D": "Check the cluster event log: fix the init script or cloud quota / instance type / permissions",
  "E": "Data skew: salt the hot key (AQE skew handling helps JOINS, not groupBy) and look for spill",
}
for k, v in ERRORS.items(): print(k, "→", v)
print()
for k, v in FIXES.items(): print(k, "=", v)

In [ ]:
# TODO 7c: match every error (E1..E5) to the best fix (A..E)
submit("7c", {"E1": "<FILL-IN 1>", "E2": "<FILL-IN 2>", "E3": "<FILL-IN 3>", "E4": "<FILL-IN 4>", "E5": "<FILL-IN 5>"})

In [ ]:
check("7c")

## 7d · Pick the right compute (exam section 1)

In [ ]:
SCENARIOS = {
  "S1": "Many business analysts run ad-hoc SQL all day on curated Delta tables; fast start, many users, control cost",
  "S2": "A nightly scheduled ETL notebook job in production; cheapest reliable option, no idle cost",
  "S3": "A data engineer explores data and develops notebooks interactively with teammates",
  "S4": "A production job that must start in seconds with no cluster management at all",
}
COMPUTE = {
  "W": "Serverless SQL warehouse (autoscaling)",
  "J": "Job cluster (created for the run, terminates after it)",
  "P": "All-purpose (interactive) cluster",
  "S": "Serverless compute for jobs",
}
for k, v in SCENARIOS.items(): print(k, "→", v)
print()
for k, v in COMPUTE.items(): print(k, "=", v)

In [ ]:
# TODO 7d: match each scenario to the best compute
submit("7d", {"S1": "<FILL-IN 1>", "S2": "<FILL-IN 2>", "S3": "<FILL-IN 3>", "S4": "<FILL-IN 4>"})

In [ ]:
check("7d")